In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import exposure

import zarr
from pylibCZIrw import czi as pyczi

In [ ]:
"""Skip during test
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Intermediate")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Proximal")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Distal")
section_folders = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        section_folders.append(folder)
"""

In [2]:
from atlas.io import is_there_a_single_tif, extract_s_number

In [3]:
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd")

#TODO: important change this to metadata readout
pixel_size = {
    'Value': 0.020,
    'Axial': 0.20,
    'Unit': 'µm'
}
""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""

tif_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_file() and folder.name.endswith(".tiff"):
        print(f"fould series image: {folder.name}")
        tif_list.append(folder)

fould series image: stitched_image_S_1.tiff
fould series image: stitched_image_S_10.tiff
fould series image: stitched_image_S_11.tiff
fould series image: stitched_image_S_12.tiff
fould series image: stitched_image_S_13.tiff
fould series image: stitched_image_S_14.tiff
fould series image: stitched_image_S_15.tiff
fould series image: stitched_image_S_16.tiff
fould series image: stitched_image_S_17.tiff
fould series image: stitched_image_S_18.tiff
fould series image: stitched_image_S_19.tiff
fould series image: stitched_image_S_2.tiff
fould series image: stitched_image_S_20.tiff
fould series image: stitched_image_S_21.tiff
fould series image: stitched_image_S_22.tiff
fould series image: stitched_image_S_23.tiff
fould series image: stitched_image_S_24.tiff
fould series image: stitched_image_S_25.tiff
fould series image: stitched_image_S_26.tiff
fould series image: stitched_image_S_27.tiff
fould series image: stitched_image_S_28.tiff
fould series image: stitched_image_S_29.tiff
fould series

In [4]:
"""SKIP during test
tif_list = []
for section_folder in section_folders:
    single_tif, tif_path = is_there_a_single_tif(section_folder)
    if single_tif:
        #print(tif_path)
        tif_list.append(tif_path)
"""

tif_list_sorted = sorted(tif_list, key=extract_s_number)

In [5]:
from atlas.io.fibics_metadata import extract_tif_metadata, get_pixel_size_from_tif

In [6]:
"""SKIP during test
#TODO: important change this to metadata readout
pixel_size = {
    'Value': get_pixel_size_from_tif(tif_list_sorted[0].name, tif_list_sorted[0].parent),
    'Axial': 0.07,
    'Unit': 'µm'
}
"""

""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""
print("work on pixel size")

work on pixel size


In [7]:
from atlas.io import create_empty_folder, rm_tree
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation, rescale_image_intensity
from atlas.alignment import calculate_cumulative_shifts, initialize_alignment_df, pairwise_alignment

In [ ]:
output_path = series_folder.joinpath("alignment_results")
create_empty_folder(output_path)

down_scale = 10
tif_list_sorted = sorted(tif_list, key=extract_s_number)

# initialize the dataframe, in particular we asign the pair-wise patching of the images
# based on the sorted list of tiff.
z_align_df = initialize_alignment_df(tif_list_sorted, down_scale)
# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_alignment(z_align_df)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# ✅ At the end, save the DataFrame as a CSV for later analysis
z_align_df_path = output_path.joinpath("z_alignment_results.pkl")
z_align_df.to_pickle(z_align_df_path)

z_align_df 

Created new (or emptied) folder: Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd\alignment_results
Processing alignment: ref -> stitched_image_S_1.tiff, moving -> stitched_image_S_1.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [0. 0.]
current shift: [0. 0.]
Processing alignment: ref -> stitched_image_S_1.tiff, moving -> stitched_image_S_2.tiff
crop pixel shift: [-39 150]
Detected pixel offset based on crops (row, col): [2. 6.]
current shift: [-37. 156.]
Processing alignment: ref -> stitched_image_S_2.tiff, moving -> stitched_image_S_3.tiff
crop pixel shift: [0 2]
Detected pixel offset based on crops (row, col): [-10.  -2.]
current shift: [-10.   0.]
Processing alignment: ref -> stitched_image_S_3.tiff, moving -> stitched_image_S_4.tiff
crop pixel shift: [ 0 -2]
Detected pixel offset based on crops (row, col): [ 2. -4.]
current shift: [ 2. -6.]
Processing alignment: ref -> stitched_image_S_4.tiff, moving -> stitched_image_S_5.tiff
crop pi

TypeError: NDFrame.to_pickle() got an unexpected keyword argument 'index'

In [12]:
from atlas.io import apply_alignment, zarr_array_to_czi
# ✅ At the end, save the DataFrame as a CSV for later analysis

z_align_df = pd.read_pickle(z_align_df_path)
z_align_df

,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(232, 874, 84, 1952)","(232, 874, 84, 1952)","[0.0, 0.0]","(232, 874, 84, 1952)","[0.0, 0.0]",10
1,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(82, 727, 123, 1994)","(232, 874, 84, 1952)","[-37.0, 156.0]","(238, 883, 86, 1957)","[-37.0, 156.0]",10
2,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(80, 723, 123, 1993)","(82, 727, 123, 1994)","[-10.0, 0.0]","(236, 879, 76, 1946)","[-47.0, 156.0]",10
3,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(82, 726, 123, 1993)","(80, 723, 123, 1993)","[2.0, -6.0]","(232, 876, 78, 1948)","[-45.0, 150.0]",10
4,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(80, 723, 123, 1991)","(82, 726, 123, 1993)","[2.0, -6.0]","(224, 867, 80, 1948)","[-43.0, 144.0]",10
...,...,...,...,...,...,...,...,...
90,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(82, 725, 123, 1987)","(82, 731, 123, 1986)","[6.0, -2.0]","(186, 829, 119, 1983)","[-4.0, 104.0]",10
91,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(82, 733, 123, 1993)","(82, 725, 123, 1987)","[-28.0, 13.0]","(199, 850, 91, 1961)","[-32.0, 117.0]",10
92,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(82, 730, 123, 1987)","(82, 733, 123, 1993)","[19.0, -4.0]","(195, 843, 110, 1974)","[-13.0, 113.0]",10
93,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10...,"(82, 732, 123, 1977)","(82, 730, 123, 1987)","[-1.0, -8.0]","(187, 837, 109, 1963)","[-14.0, 105.0]",10


In [ ]:
# drop sections that are out of place, I need a better procedure for this, in this case easy because they were at the end
# Drop the last 2 rows in-place
#z_align_df.drop(z_align_df.index[-2:], inplace=True)
# TODO: work on a code to drop based on index values, however this might require re-run of alignment.
# a solution would be to uncouple the pairwise alignment from the comulative one, so this would not require
# a full rerun but just a couple of extra operations

In [11]:
use_down_sample = True
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

we will downsample during saving of the zarr, this is good during testing for visual inspection
shape of the zarr array to create: (1976, 740, 95)
creating zarr at: Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd\L32-10-1ROI-w1-20nm-bsd.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18
Done for idx 19
Done for idx 20
Done for idx 21
Done for idx 22
Done for idx 23
Done for idx 24
Done for idx 25
Done for idx 26
Done for idx 27
Done for idx 28
Done for idx 29
Done for idx 30
Done for idx 31
Done for idx 32
Done for idx 33
Done for idx 34
Done for idx 35
Done for idx 36
Done for idx 37
Done for idx 38
Done for idx 39
Done for idx 40
Done for idx 41
Done for idx 42
Done for idx 43
Done for idx 44
Done for idx 45
Done for idx 46


In [13]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

Saving CZI file to: Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd\L32-10-1ROI-w1-20nm-bsd_ds_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Process

In [14]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [15]:
use_down_sample = False
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

doing full scale alignment, it is recomended to check results before with downscale
shape of the zarr array to create: (19400, 7040, 95)
creating zarr at: Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd\L32-10-1ROI-w1-20nm-bsd.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18
Done for idx 19
Done for idx 20
Done for idx 21
Done for idx 22
Done for idx 23
Done for idx 24
Done for idx 25
Done for idx 26
Done for idx 27
Done for idx 28
Done for idx 29
Done for idx 30
Done for idx 31
Done for idx 32
Done for idx 33
Done for idx 34
Done for idx 35
Done for idx 36
Done for idx 37
Done for idx 38
Done for idx 39
Done for idx 40
Done for idx 41
Done for idx 42
Done for idx 43
Done for idx 44
Done for idx 45
Done for idx 46
Done for i

In [16]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

Saving CZI file to: Z:\zeinab\ATLAS-projects\atlas_L32-10-1\L32-10-1ROI-w1-20nm-bsd\L32-10-1ROI-w1-20nm-bsd_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing